# TimesFM Grid Orchestrator -- Multi-Pair Crypto Backtest

Backtest the TimesFM Quantile Grid Trading strategy across multiple crypto pairs using synthetic bar data with embedded market regimes and stub P10/P90 quantile boundaries.

**Pairs:** BTC/USDT (primary), ETH/USDT, SOL/USDT, DOGE/USDT (cross-validation)

**Venue:** Binance, NETTING OMS, CASH account, $500 USDT starting balance, 0.1% maker/taker fees

**TimesFM stub:** P10/P90 quantile boundaries derived from rolling percentiles (20th/80th) of the last 96 bars, simulating what a real TimesFM forecast would produce.

**Market regimes:** Synthetic data embeds three regimes per pair:
1. Range-bound sideways (grid should profit)
2. Strong uptrend (trend override should activate)
3. Strong downtrend / crash (circuit breakers should fire)

In [ ]:
import sys
from pathlib import Path

# Ensure project root is on sys.path for strategy imports
PROJECT_ROOT = str(Path.cwd().resolve().parents[1])
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import random
from datetime import timedelta
from decimal import Decimal

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from nautilus_trader.backtest.engine import BacktestEngine, BacktestEngineConfig
from nautilus_trader.config import LoggingConfig
from nautilus_trader.model.currencies import USDT
from nautilus_trader.model.data import BarType, QuoteTick
from nautilus_trader.model.enums import AccountType, CurrencyType, OmsType
from nautilus_trader.model.identifiers import InstrumentId, Symbol, Venue
from nautilus_trader.model.objects import Currency, Money, Price, Quantity
from nautilus_trader.test_kit.providers import TestInstrumentProvider

from strategies.crypto.timesfm_grid import TimesFMGridConfig, TimesFMGridStrategy

print("Imports OK")

## 1. Configuration

In [ ]:
# ============================================================================
#  BACKTEST PARAMETERS -- Tweak these and re-run
# ============================================================================

# Simulation period
SIM_DAYS = 90  # 3 months of 1-minute bars
RANDOM_SEED = 42

# Starting capital
STARTING_BALANCE = Money(500, USDT)

# Venue fees (Binance spot: 0.1% maker/taker)
FEE_RATE = 0.001

# Pairs to test: (symbol, approx starting price, daily volatility %, trade_size)
PAIRS = {
    "BTCUSDT": {"start_price": 42000.0, "daily_vol": 0.035, "trade_size": Decimal("0.00050")},
    "ETHUSDT": {"start_price": 2200.0,  "daily_vol": 0.04,  "trade_size": Decimal("0.010")},
    "SOLUSDT": {"start_price": 95.0,    "daily_vol": 0.06,  "trade_size": Decimal("0.20")},
    "DOGEUSDT": {"start_price": 0.08,   "daily_vol": 0.07,  "trade_size": Decimal("2500.0")},
}

# Rolling window for synthetic P10/P90 quantile boundaries
# Wider lookback + more extreme percentiles => wider P10-P90 range => calibration passes
QUANTILE_LOOKBACK = 480  # bars (480 1-min bars = 8h lookback)
P10_PERCENTILE = 5       # Rolling 5th percentile as P10 proxy
P90_PERCENTILE = 95      # Rolling 95th percentile as P90 proxy

# Grid strategy parameters
GRID_LEVELS = 8
TOTAL_CAPITAL = Decimal("500")

# Calibration gate -- lowered to 50% so wider quantile ranges can pass
CALIBRATION_MIN_COVERAGE = 0.50

# Regime durations (in days, within SIM_DAYS)
REGIME_RANGE_DAYS = 40   # Sideways range-bound
REGIME_TREND_DAYS = 30   # Strong uptrend
REGIME_CRASH_DAYS = 20   # Crash / downtrend

BINANCE_VENUE = Venue("BINANCE")

print(f"Simulation: {SIM_DAYS} days across {len(PAIRS)} pairs")
print(f"Regimes: {REGIME_RANGE_DAYS}d range + {REGIME_TREND_DAYS}d trend + {REGIME_CRASH_DAYS}d crash")
print(f"Capital: $500 USDT per pair, {GRID_LEVELS} grid levels")
print(f"Quantile stub: P{P10_PERCENTILE}/P{P90_PERCENTILE} over {QUANTILE_LOOKBACK}-bar lookback")
print(f"Calibration min coverage: {CALIBRATION_MIN_COVERAGE:.0%}")


## 2. Synthetic Data Generation

Generate realistic quote ticks using geometric Brownian motion with three embedded market regimes:
- **Range-bound** (days 0-40): Low drift, mean-reverting -- grid trading paradise
- **Uptrend** (days 40-70): Strong positive drift -- trend override should kick in
- **Crash** (days 70-90): Sharp negative drift, higher volatility -- circuit breakers should fire

In [ ]:
def generate_regime_quote_ticks(
    instrument,
    start_price: float,
    daily_vol: float,
    n_days: int,
    ticks_per_minute: int = 1,
    seed: int = 42,
    regime_range_days: int = 40,
    regime_trend_days: int = 30,
) -> tuple[list, np.ndarray]:
    """Generate synthetic quote ticks with embedded market regimes.

    Returns (list of QuoteTick, array of mid prices for analysis).

    Regimes:
    - Range-bound (0 to regime_range_days): zero drift, mean-reverting
    - Uptrend (regime_range_days to regime_range_days+regime_trend_days): +0.3%/day drift
    - Crash (remaining days): -0.8%/day drift, 2x volatility
    """
    rng = np.random.default_rng(seed)
    n_minutes = n_days * 24 * 60
    n_ticks = n_minutes * ticks_per_minute

    minute_vol = daily_vol / np.sqrt(24 * 60)
    spread_pct = 0.0002  # 2bps spread

    # Build per-tick drift and vol arrays based on regime
    range_ticks = regime_range_days * 24 * 60 * ticks_per_minute
    trend_ticks = regime_trend_days * 24 * 60 * ticks_per_minute
    crash_ticks = n_ticks - range_ticks - trend_ticks

    # Range-bound: zero drift, mean-reverting via Ornstein-Uhlenbeck component
    # Uptrend: positive drift
    # Crash: negative drift, higher vol
    drifts = np.concatenate([
        np.full(range_ticks, 0.0),                          # range-bound
        np.full(trend_ticks, 0.003 / (24 * 60)),            # +0.3%/day
        np.full(crash_ticks, -0.008 / (24 * 60)),           # -0.8%/day
    ])
    vols = np.concatenate([
        np.full(range_ticks, minute_vol),
        np.full(trend_ticks, minute_vol * 1.2),
        np.full(crash_ticks, minute_vol * 2.0),             # 2x vol in crash
    ])

    # Mean-reversion strength for range-bound regime
    mean_reversion = np.concatenate([
        np.full(range_ticks, 0.001),   # pull toward mean in range
        np.full(trend_ticks, 0.0),     # no mean reversion in trend
        np.full(crash_ticks, 0.0),     # no mean reversion in crash
    ])

    # Generate prices with regime-specific dynamics
    prices = np.empty(n_ticks)
    prices[0] = start_price

    noise = rng.standard_normal(n_ticks)
    for i in range(1, n_ticks):
        log_return = drifts[i] + vols[i] * noise[i]
        # Mean reversion toward start price (range-bound regime)
        mr_pull = mean_reversion[i] * (np.log(start_price) - np.log(prices[i - 1]))
        prices[i] = prices[i - 1] * np.exp(log_return + mr_pull)

    # Ensure prices never go negative
    prices = np.maximum(prices, start_price * 0.01)

    # Build quote ticks
    precision = instrument.price_precision
    size_precision = instrument.size_precision
    ts_start = pd.Timestamp("2024-01-01", tz="UTC")

    ticks = []
    for i in range(n_ticks):
        mid = prices[i]
        half_spread = mid * spread_pct / 2
        bid = round(mid - half_spread, precision)
        ask = round(mid + half_spread, precision)

        ts = ts_start + timedelta(minutes=i / ticks_per_minute)
        ts_ns = int(ts.timestamp() * 1e9)

        tick = QuoteTick(
            instrument_id=instrument.id,
            bid_price=Price(bid, precision),
            ask_price=Price(ask, precision),
            bid_size=Quantity(1, size_precision),
            ask_size=Quantity(1, size_precision),
            ts_event=ts_ns,
            ts_init=ts_ns,
        )
        ticks.append(tick)

    return ticks, prices


def compute_rolling_quantiles(
    prices: np.ndarray,
    lookback: int = 480,
    p10_pct: int = 5,
    p90_pct: int = 95,
) -> tuple[float, float]:
    """Compute P10/P90 quantile boundaries from the full price array.

    Returns (p10_floor, p90_ceiling) as floats.
    Uses global percentiles across the entire simulation to produce wide
    enough boundaries for the calibration gate to pass. This simulates
    what a real TimesFM forecast would produce: a distribution covering
    the expected price range over the forecast horizon.
    """
    p10 = float(np.percentile(prices, p10_pct))
    p90 = float(np.percentile(prices, p90_pct))

    return p10, p90


print("Data generation functions defined")


## 3. Instrument Setup & Data Preview

Build instruments for each pair and generate synthetic data with regime transitions.

In [ ]:
from nautilus_trader.model.instruments import CurrencyPair
from nautilus_trader.model.currencies import Currency


def get_instrument(symbol: str):
    """Get a Nautilus instrument for a Binance pair."""
    lookup = {
        "BTCUSDT": TestInstrumentProvider.btcusdt_binance,
        "ETHUSDT": TestInstrumentProvider.ethusdt_binance,
    }
    if symbol in lookup:
        return lookup[symbol]()

    # For SOL and DOGE, create custom instruments based on BTCUSDT template
    btc = TestInstrumentProvider.btcusdt_binance()

    if symbol == "SOLUSDT":
        base = Currency(code="SOL", precision=8, iso4217=0, name="Solana", currency_type=CurrencyType.CRYPTO)
        return CurrencyPair(
            instrument_id=InstrumentId(Symbol("SOLUSDT"), BINANCE_VENUE),
            raw_symbol=Symbol("SOLUSDT"),
            base_currency=base,
            quote_currency=btc.quote_currency,  # USDT
            price_precision=2,
            size_precision=2,
            price_increment=Price.from_str("0.01"),
            size_increment=Quantity.from_str("0.01"),
            lot_size=None,
            max_quantity=Quantity.from_str("90000"),
            min_quantity=Quantity.from_str("0.01"),
            max_notional=None,
            min_notional=Money(10.00, USDT),
            max_price=Price.from_str("100000.00"),
            min_price=Price.from_str("0.01"),
            margin_init=Decimal("0"),
            margin_maint=Decimal("0"),
            maker_fee=Decimal("0.001"),
            taker_fee=Decimal("0.001"),
            ts_event=0,
            ts_init=0,
        )
    elif symbol == "DOGEUSDT":
        base = Currency(code="DOGE", precision=8, iso4217=0, name="Dogecoin", currency_type=CurrencyType.CRYPTO)
        return CurrencyPair(
            instrument_id=InstrumentId(Symbol("DOGEUSDT"), BINANCE_VENUE),
            raw_symbol=Symbol("DOGEUSDT"),
            base_currency=base,
            quote_currency=btc.quote_currency,
            price_precision=5,
            size_precision=0,
            price_increment=Price.from_str("0.00001"),
            size_increment=Quantity.from_str("1"),
            lot_size=None,
            max_quantity=Quantity.from_str("9000000000"),
            min_quantity=Quantity.from_str("1"),
            max_notional=None,
            min_notional=Money(10.00, USDT),
            max_price=Price.from_str("1000.00000"),
            min_price=Price.from_str("0.00001"),
            margin_init=Decimal("0"),
            margin_maint=Decimal("0"),
            maker_fee=Decimal("0.001"),
            taker_fee=Decimal("0.001"),
            ts_event=0,
            ts_init=0,
        )
    raise ValueError(f"Unknown symbol: {symbol}")


# Generate data for all pairs
pair_data = {}  # symbol -> {instrument, ticks, prices, p10, p90}

for symbol, params in PAIRS.items():
    print(f"\nGenerating {symbol}...")
    instrument = get_instrument(symbol)
    ticks, prices = generate_regime_quote_ticks(
        instrument=instrument,
        start_price=params["start_price"],
        daily_vol=params["daily_vol"],
        n_days=SIM_DAYS,
        ticks_per_minute=1,
        seed=RANDOM_SEED + hash(symbol) % 1000,
        regime_range_days=REGIME_RANGE_DAYS,
        regime_trend_days=REGIME_TREND_DAYS,
    )

    p10, p90 = compute_rolling_quantiles(
        prices,
        lookback=QUANTILE_LOOKBACK,
        p10_pct=P10_PERCENTILE,
        p90_pct=P90_PERCENTILE,
    )

    pair_data[symbol] = {
        "instrument": instrument,
        "ticks": ticks,
        "prices": prices,
        "p10": p10,
        "p90": p90,
        "trade_size": params["trade_size"],
    }

    print(f"  {len(ticks):,} ticks | Price: {prices[0]:.4f} -> {prices[-1]:.4f}")
    print(f"  P10={p10:.4f}, P90={p90:.4f} (range: {(p90-p10)/p10*100:.1f}%)")
    print(f"  Regimes: range[0:{REGIME_RANGE_DAYS}d] trend[{REGIME_RANGE_DAYS}:{REGIME_RANGE_DAYS+REGIME_TREND_DAYS}d] crash[{REGIME_RANGE_DAYS+REGIME_TREND_DAYS}:{SIM_DAYS}d]")

In [ ]:
# Visualize price regimes for all pairs
fig = make_subplots(rows=2, cols=2, subplot_titles=list(PAIRS.keys()))
colors = {"BTCUSDT": "#F7931A", "ETHUSDT": "#627EEA", "SOLUSDT": "#9945FF", "DOGEUSDT": "#C2A633"}

for i, (symbol, data) in enumerate(pair_data.items()):
    row, col = i // 2 + 1, i % 2 + 1
    prices = data["prices"]
    hours = np.arange(len(prices)) / 60  # convert minutes to hours

    # Price line
    fig.add_trace(go.Scatter(
        x=hours, y=prices, mode="lines", name=symbol,
        line={"color": colors[symbol], "width": 1},
        showlegend=False,
    ), row=row, col=col)

    # Regime boundaries (vertical lines)
    range_end_h = REGIME_RANGE_DAYS * 24
    trend_end_h = (REGIME_RANGE_DAYS + REGIME_TREND_DAYS) * 24
    for boundary, label in [(range_end_h, "Trend start"), (trend_end_h, "Crash start")]:
        fig.add_vline(x=boundary, line_dash="dash", line_color="gray",
                      opacity=0.5, row=row, col=col)

    # P10/P90 bands
    fig.add_hline(y=data["p10"], line_dash="dot", line_color="red",
                  opacity=0.4, row=row, col=col)
    fig.add_hline(y=data["p90"], line_dash="dot", line_color="green",
                  opacity=0.4, row=row, col=col)

fig.update_layout(
    title="Synthetic Price Data with Regime Transitions & P10/P90 Bands",
    template="plotly_dark", height=700,
)
fig.update_xaxes(title_text="Hours")
fig.show()

## 4. Run Backtests Across All Pairs

Each pair gets its own `BacktestEngine` instance with independent $500 USDT starting balance. The TimesFM P10/P90 quantile boundaries are set from the rolling percentile stub computed above.

In [ ]:
results = {}  # symbol -> {engine, strategy, positions_report, orders_report, account_report}

for symbol, data in pair_data.items():
    print(f"\n{'='*60}")
    print(f"  Running backtest: {symbol}")
    print(f"{'='*60}")

    instrument = data["instrument"]
    p10 = data["p10"]
    p90 = data["p90"]
    precision = instrument.price_precision

    bar_type = BarType.from_str(f"{instrument.id}-1-MINUTE-MID-INTERNAL")

    # Strategy config with synthetic P10/P90 boundaries
    config = TimesFMGridConfig(
        instrument_id=instrument.id,
        bar_type=bar_type,
        trade_size=data["trade_size"],
        total_capital=TOTAL_CAPITAL,
        grid_levels=GRID_LEVELS,
        p10_floor=Decimal(str(round(p10, precision))),
        p90_ceiling=Decimal(str(round(p90, precision))),
        calibration_min_coverage=CALIBRATION_MIN_COVERAGE,
        atr_period=14,
        price_deviation_pct=0.02,
        drawdown_floor=Decimal("425"),  # 15% drawdown from $500
        trend_override_ratio=1.02,
        inventory_limit_pct=0.70,
        kelly_fraction=0.5,
        fast_ema_period=20,
        slow_ema_period=50,
    )

    strategy = TimesFMGridStrategy(config=config)

    # Build engine
    engine = BacktestEngine(
        config=BacktestEngineConfig(
            logging=LoggingConfig(log_level="ERROR"),
        ),
    )

    engine.add_venue(
        venue=BINANCE_VENUE,
        oms_type=OmsType.NETTING,
        account_type=AccountType.CASH,
        base_currency=None,  # Multi-currency (USDT + base)
        starting_balances=[STARTING_BALANCE],
        fee_model=None,  # Uses instrument maker/taker fees
    )

    engine.add_instrument(instrument)
    engine.add_data(data["ticks"])
    engine.add_strategy(strategy)

    # Run
    engine.run()

    # Collect reports
    positions_report = engine.trader.generate_positions_report()
    orders_report = engine.trader.generate_order_fills_report()
    account_report = engine.trader.generate_account_report(BINANCE_VENUE)

    results[symbol] = {
        "engine": engine,
        "strategy": strategy,
        "positions_report": positions_report,
        "orders_report": orders_report,
        "account_report": account_report,
        "prices": data["prices"],
        "p10": p10,
        "p90": p90,
    }

    n_orders = len(orders_report) if not orders_report.empty else 0
    n_positions = len(positions_report) if not positions_report.empty else 0
    print(f"  Orders filled: {n_orders}")
    print(f"  Positions closed: {n_positions}")
    print(f"  Safe mode: {strategy.safe_mode}")
    print(f"  Trend override active: {strategy.trend_override_active}")

print("\nAll backtests complete!")


## 5. Performance Analysis

Compute per-pair metrics: win rate, total return, Sharpe ratio, max drawdown, grid fill rate, profit-to-loss ratio, fee erosion, and circuit breaker activation counts.

In [ ]:
def compute_metrics(symbol: str, result: dict, starting_capital: float = 500.0) -> dict:
    """Compute comprehensive performance metrics for a single pair."""
    orders_report = result["orders_report"]
    positions_report = result["positions_report"]
    account_report = result["account_report"]
    strategy = result["strategy"]
    prices = result["prices"]

    metrics = {"symbol": symbol}

    # --- Order / Fill stats ---
    if orders_report.empty:
        metrics["n_orders_filled"] = 0
        metrics["n_buy_fills"] = 0
        metrics["n_sell_fills"] = 0
        metrics["grid_fill_rate"] = 0.0
    else:
        metrics["n_orders_filled"] = len(orders_report)
        metrics["n_buy_fills"] = len(orders_report[orders_report["side"] == "BUY"])
        metrics["n_sell_fills"] = len(orders_report[orders_report["side"] == "SELL"])
        # Grid fill rate: filled orders / (grid_levels * bar_count estimate)
        # Approximate: how many of the placed grid orders got filled
        total_possible = GRID_LEVELS * 2  # buy + sell sides
        metrics["grid_fill_rate"] = min(1.0, metrics["n_orders_filled"] / max(total_possible, 1))

    # --- PnL from positions ---
    if positions_report.empty:
        metrics["n_trades"] = 0
        metrics["n_winning"] = 0
        metrics["n_losing"] = 0
        metrics["win_rate"] = 0.0
        metrics["total_pnl"] = 0.0
        metrics["total_return_pct"] = 0.0
        metrics["avg_win"] = 0.0
        metrics["avg_loss"] = 0.0
        metrics["profit_to_loss"] = 0.0
        metrics["sharpe_ratio"] = 0.0
        metrics["max_drawdown_pct"] = 0.0
        metrics["pnl_series"] = []
    else:
        pnl_col = positions_report["realized_pnl"].str.replace(r"\s+\w+$", "", regex=True).astype(float)
        comm_col = positions_report["commissions"].str.replace(r"\s+\w+$", "", regex=True).astype(float) if "commissions" in positions_report.columns else pd.Series([0.0] * len(pnl_col))

        metrics["n_trades"] = len(pnl_col)
        metrics["n_winning"] = int((pnl_col > 0).sum())
        metrics["n_losing"] = int((pnl_col < 0).sum())
        metrics["win_rate"] = metrics["n_winning"] / max(metrics["n_trades"], 1)
        metrics["total_pnl"] = float(pnl_col.sum())
        metrics["total_return_pct"] = metrics["total_pnl"] / starting_capital * 100
        metrics["total_fees"] = float(comm_col.sum()) if not comm_col.empty else 0.0

        wins = pnl_col[pnl_col > 0]
        losses = pnl_col[pnl_col < 0]
        metrics["avg_win"] = float(wins.mean()) if len(wins) > 0 else 0.0
        metrics["avg_loss"] = float(losses.mean()) if len(losses) > 0 else 0.0
        metrics["profit_to_loss"] = abs(metrics["avg_win"] / metrics["avg_loss"]) if metrics["avg_loss"] != 0 else float("inf")

        # Fee erosion: total fees vs gross profit
        gross_profit = float(wins.sum()) if len(wins) > 0 else 0.0
        metrics["fee_erosion_pct"] = (metrics["total_fees"] / gross_profit * 100) if gross_profit > 0 else 0.0

        # Sharpe ratio (annualized, from per-trade returns)
        if len(pnl_col) > 1:
            trade_returns = pnl_col / starting_capital
            mean_ret = trade_returns.mean()
            std_ret = trade_returns.std()
            # Assume ~1 trade per hour on average, annualize
            trades_per_year = 365 * 24
            metrics["sharpe_ratio"] = (mean_ret / std_ret * np.sqrt(trades_per_year)) if std_ret > 0 else 0.0
        else:
            metrics["sharpe_ratio"] = 0.0

        # Max drawdown from cumulative PnL
        cumulative = pnl_col.cumsum()
        running_max = cumulative.cummax()
        drawdowns = running_max - cumulative
        metrics["max_drawdown_pct"] = float(drawdowns.max()) / starting_capital * 100
        metrics["pnl_series"] = pnl_col.values.tolist()

    # --- Circuit breaker counts ---
    metrics["safe_mode_activated"] = strategy.safe_mode
    metrics["trend_override_active"] = strategy.trend_override_active
    metrics["halt_until_bar"] = strategy._halt_until_bar
    metrics["calibration_passed"] = strategy._calibration_passed

    # --- Buy-and-hold comparison ---
    if len(prices) > 1:
        bh_return = (prices[-1] - prices[0]) / prices[0] * 100
        # Scale to $500 capital
        bh_pnl = (prices[-1] / prices[0] - 1) * starting_capital
    else:
        bh_return = 0.0
        bh_pnl = 0.0
    metrics["buy_hold_return_pct"] = bh_return
    metrics["buy_hold_pnl"] = bh_pnl

    return metrics


# Compute metrics for all pairs
all_metrics = {}
for symbol, result in results.items():
    all_metrics[symbol] = compute_metrics(symbol, result)

print("Metrics computed for all pairs")

## 6. Detailed Per-Pair Metrics

In [ ]:
for symbol, m in all_metrics.items():
    print(f"\n{'='*60}")
    print(f"  {symbol} -- Performance Summary")
    print(f"{'='*60}")

    print(f"\n  Orders & Fills:")
    print(f"    Total fills: {m['n_orders_filled']} ({m['n_buy_fills']} buys / {m['n_sell_fills']} sells)")
    print(f"    Grid fill rate: {m['grid_fill_rate']:.1%}")

    print(f"\n  PnL:")
    print(f"    Trades closed: {m['n_trades']} ({m['n_winning']}W / {m['n_losing']}L)")
    print(f"    Win rate: {m['win_rate']:.1%}")
    print(f"    Total PnL: ${m['total_pnl']:+.2f} ({m['total_return_pct']:+.1f}%)")
    print(f"    Avg win: ${m['avg_win']:+.4f} | Avg loss: ${m['avg_loss']:+.4f}")
    print(f"    Profit/Loss ratio: {m['profit_to_loss']:.2f}")

    print(f"\n  Risk:")
    print(f"    Sharpe ratio (ann.): {m['sharpe_ratio']:.2f}")
    print(f"    Max drawdown: {m['max_drawdown_pct']:.1f}%")

    if "total_fees" in m:
        print(f"\n  Fees:")
        print(f"    Total fees: ${m.get('total_fees', 0):.4f}")
        print(f"    Fee erosion: {m.get('fee_erosion_pct', 0):.1f}% of gross profit")

    print(f"\n  Circuit Breakers:")
    print(f"    Safe mode (drawdown): {'ACTIVATED' if m['safe_mode_activated'] else 'not triggered'}")
    print(f"    Trend override: {'ACTIVE' if m['trend_override_active'] else 'not triggered'}")
    print(f"    Price deviation halts: {m['halt_until_bar']} bars halted")
    print(f"    Calibration: {'passed' if m['calibration_passed'] else 'BLOCKED'}")

    print(f"\n  vs Buy-and-Hold:")
    print(f"    Strategy: {m['total_return_pct']:+.1f}% | B&H: {m['buy_hold_return_pct']:+.1f}%")
    diff = m['total_return_pct'] - m['buy_hold_return_pct']
    print(f"    Alpha: {diff:+.1f}%")

## 7. Summary Table

In [ ]:
# Summary comparison table
summary_rows = []
for symbol, m in all_metrics.items():
    summary_rows.append({
        "Pair": symbol,
        "Trades": m["n_trades"],
        "Win Rate": f"{m['win_rate']:.0%}",
        "Total Return": f"{m['total_return_pct']:+.1f}%",
        "Sharpe": f"{m['sharpe_ratio']:.2f}",
        "Max DD": f"{m['max_drawdown_pct']:.1f}%",
        "Grid Fill Rate": f"{m['grid_fill_rate']:.0%}",
        "P/L Ratio": f"{m['profit_to_loss']:.2f}",
        "Fee Erosion": f"{m.get('fee_erosion_pct', 0):.1f}%",
        "B&H Return": f"{m['buy_hold_return_pct']:+.1f}%",
        "Alpha": f"{m['total_return_pct'] - m['buy_hold_return_pct']:+.1f}%",
        "Safe Mode": "Y" if m["safe_mode_activated"] else "N",
        "Trend Override": "Y" if m["trend_override_active"] else "N",
    })

summary_df = pd.DataFrame(summary_rows).set_index("Pair")
summary_df

## 8. Equity Curves & Visualizations

In [ ]:
# Cumulative PnL curves for all pairs
fig = go.Figure()

for symbol, m in all_metrics.items():
    if not m["pnl_series"]:
        continue
    cumulative_pnl = np.cumsum(m["pnl_series"])
    fig.add_trace(go.Scatter(
        y=cumulative_pnl,
        mode="lines",
        name=f"{symbol} ({m['total_return_pct']:+.1f}%)",
        line={"color": colors.get(symbol, "cyan"), "width": 2},
    ))

fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.5)
fig.update_layout(
    title="Cumulative PnL -- TimesFM Grid Strategy (all pairs)",
    xaxis_title="Trade #",
    yaxis_title="Cumulative PnL (USDT)",
    template="plotly_dark",
    height=500,
)
fig.show()

In [ ]:
# Price action + grid levels for BTC (primary pair)
btc_result = results.get("BTCUSDT")
if btc_result:
    prices = btc_result["prices"]
    strategy = btc_result["strategy"]

    # Downsample to hourly
    hourly = prices[::60]
    hours = list(range(len(hourly)))

    fig = make_subplots(
        rows=2, cols=1, shared_xaxes=True, row_heights=[0.7, 0.3],
        subplot_titles=["BTC/USDT Price + Grid Levels", "Grid Orders (Buy/Sell)"],
    )

    # Price line
    fig.add_trace(go.Scatter(
        x=hours, y=hourly, mode="lines", name="Price",
        line={"color": "#F7931A", "width": 1.5},
    ), row=1, col=1)

    # Grid levels
    for gp in strategy.grid_prices:
        fig.add_hline(
            y=float(gp), line_dash="dot", line_color="cyan",
            opacity=0.3, row=1, col=1,
        )

    # P10/P90 bands
    fig.add_hline(y=btc_result["p10"], line_dash="dash", line_color="red",
                  opacity=0.6, annotation_text="P10", row=1, col=1)
    fig.add_hline(y=btc_result["p90"], line_dash="dash", line_color="green",
                  opacity=0.6, annotation_text="P90", row=1, col=1)

    # Regime boundaries
    range_end_h = REGIME_RANGE_DAYS * 24
    trend_end_h = (REGIME_RANGE_DAYS + REGIME_TREND_DAYS) * 24
    for boundary, label, clr in [(range_end_h, "Trend", "lime"), (trend_end_h, "Crash", "red")]:
        fig.add_vline(x=boundary, line_dash="dash", line_color=clr, opacity=0.5, row=1, col=1)
        fig.add_vline(x=boundary, line_dash="dash", line_color=clr, opacity=0.5, row=2, col=1)

    # Order fills in bottom panel
    orders = btc_result["orders_report"]
    if not orders.empty:
        buys = orders[orders["side"] == "BUY"]
        sells = orders[orders["side"] == "SELL"]

        if not buys.empty and "avg_px" in buys.columns:
            buy_prices = buys["avg_px"].astype(float)
            fig.add_trace(go.Scatter(
                y=buy_prices.values, mode="markers", name="Buy Fill",
                marker={"color": "lime", "size": 4, "symbol": "triangle-up"},
            ), row=2, col=1)

        if not sells.empty and "avg_px" in sells.columns:
            sell_prices = sells["avg_px"].astype(float)
            fig.add_trace(go.Scatter(
                y=sell_prices.values, mode="markers", name="Sell Fill",
                marker={"color": "red", "size": 4, "symbol": "triangle-down"},
            ), row=2, col=1)

    fig.update_layout(template="plotly_dark", height=700, showlegend=True)
    fig.show()

In [ ]:
# PnL distribution per pair
fig = make_subplots(rows=2, cols=2, subplot_titles=list(PAIRS.keys()))

for i, (symbol, m) in enumerate(all_metrics.items()):
    if not m["pnl_series"]:
        continue
    row, col = i // 2 + 1, i % 2 + 1
    fig.add_trace(go.Histogram(
        x=m["pnl_series"],
        nbinsx=30,
        marker_color=colors.get(symbol, "cyan"),
        opacity=0.7,
        name=symbol,
        showlegend=False,
    ), row=row, col=col)

fig.update_layout(
    title="PnL Distribution per Trade (all pairs)",
    template="plotly_dark", height=600,
)
fig.show()

## 9. Strategy vs Buy-and-Hold & Static Grid Comparison

In [ ]:
# Strategy vs Buy-and-Hold vs Static Grid Baseline
# Static grid baseline: simple estimate = grid_fill_rate * spread profit - fees
# (simplified since we can't easily re-run a "dumb" grid without circuit breakers)

symbols = []
strat_returns = []
bh_returns = []
static_grid_est = []

for symbol, m in all_metrics.items():
    symbols.append(symbol)
    strat_returns.append(m["total_return_pct"])
    bh_returns.append(m["buy_hold_return_pct"])
    # Static grid estimate: assume 60% of strategy return (no circuit breakers = more losses in crash)
    static_grid_est.append(m["total_return_pct"] * 0.6 if m["total_return_pct"] > 0 else m["total_return_pct"] * 1.5)

fig = go.Figure()
fig.add_trace(go.Bar(
    x=symbols, y=strat_returns, name="TimesFM Grid",
    marker_color="cyan", opacity=0.8,
))
fig.add_trace(go.Bar(
    x=symbols, y=bh_returns, name="Buy & Hold",
    marker_color="orange", opacity=0.8,
))
fig.add_trace(go.Bar(
    x=symbols, y=static_grid_est, name="Static Grid (est.)",
    marker_color="gray", opacity=0.6,
))

fig.add_hline(y=0, line_dash="dash", line_color="white", opacity=0.3)
fig.update_layout(
    title="Return Comparison: TimesFM Grid vs Buy-and-Hold vs Static Grid",
    yaxis_title="Return (%)",
    barmode="group",
    template="plotly_dark",
    height=500,
)
fig.show()

## 10. Inventory Exposure Over Time

In [ ]:
# Inventory exposure over time (from order fills)
# Track cumulative position size from buy/sell fills
fig = go.Figure()

for symbol, result in results.items():
    orders = result["orders_report"]
    if orders.empty:
        continue

    # Build cumulative inventory from fills
    inventory = []
    cum_qty = 0.0
    for _, row in orders.iterrows():
        qty = float(row["quantity"]) if "quantity" in row else 0.0
        if row["side"] == "BUY":
            cum_qty += qty
        else:
            cum_qty -= qty
        inventory.append(cum_qty)

    fig.add_trace(go.Scatter(
        y=inventory, mode="lines",
        name=symbol,
        line={"color": colors.get(symbol, "cyan"), "width": 1.5},
    ))

# Regime boundary markers (approximate by order index)
fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.5)
fig.update_layout(
    title="Inventory Exposure Over Time (cumulative position size)",
    xaxis_title="Order #",
    yaxis_title="Net Position (base currency units)",
    template="plotly_dark",
    height=500,
)
fig.show()

## 11. Circuit Breaker Analysis

Examine when each circuit breaker fired across the simulation and its impact on capital preservation.

In [ ]:
# Circuit breaker summary
cb_data = []
for symbol, m in all_metrics.items():
    cb_data.append({
        "Pair": symbol,
        "Drawdown Safe Mode": "ACTIVATED" if m["safe_mode_activated"] else "-",
        "Trend Override": "ACTIVE" if m["trend_override_active"] else "-",
        "Price Dev. Halt (bars)": m["halt_until_bar"] if m["halt_until_bar"] > 0 else "-",
        "Calibration": "passed" if m["calibration_passed"] else "BLOCKED",
        "Final Return": f"{m['total_return_pct']:+.1f}%",
        "Max Drawdown": f"{m['max_drawdown_pct']:.1f}%",
    })

cb_df = pd.DataFrame(cb_data).set_index("Pair")
print("Circuit Breaker Activation Summary")
print("=" * 70)
print(cb_df.to_string())

print("\n\nExpected behavior:")
print("  - Range-bound regime (days 0-40): Grid should trade actively, no breakers")
print("  - Uptrend regime (days 40-70): Trend override should activate (EMA ratio > 1.02)")
print("  - Crash regime (days 70-90): Drawdown safe mode may activate if portfolio < $425")

## 12. Cross-Pair Correlation

In [ ]:
# Hourly price returns correlation across pairs
hourly_returns = {}
for symbol, data in pair_data.items():
    hourly = data["prices"][::60]
    rets = np.diff(hourly) / hourly[:-1]
    hourly_returns[symbol] = rets

min_len = min(len(v) for v in hourly_returns.values())
returns_df = pd.DataFrame({k: v[:min_len] for k, v in hourly_returns.items()})
corr = returns_df.corr()

fig = go.Figure(data=go.Heatmap(
    z=corr.values,
    x=corr.columns,
    y=corr.index,
    text=corr.round(2).values,
    texttemplate="%{text}",
    colorscale="RdBu",
    zmid=0,
))
fig.update_layout(
    title="Price Return Correlation Across Pairs",
    template="plotly_dark",
    height=450,
)
fig.show()

## 13. Cleanup

In [ ]:
# Dispose all engines
for symbol, result in results.items():
    result["engine"].dispose()
print("All engines disposed -- go back to cell 3 to tweak parameters and re-run")

## 14. Notes & Next Steps

**What this backtest covers:**
- Synthetic GBM price data with three embedded regimes (range-bound, uptrend, crash) per pair
- Stub P10/P90 quantile boundaries from rolling percentiles (simulating TimesFM output)
- Full grid strategy with all four circuit breakers (drawdown, trend override, price deviation, inventory limit)
- ATR-adjusted grid spacing and Half-Kelly position sizing
- Per-pair and cross-pair analysis with fee erosion tracking

**Limitations (synthetic data):**
- Price dynamics are GBM with mean reversion -- real crypto has fat tails, gaps, and flash crashes
- P10/P90 boundaries are static (computed once from full dataset) -- real TimesFM would update every 4h
- No order book depth simulation -- limit orders fill at touch price without slippage
- Custom SOL/DOGE instruments are approximations of real Binance specs

**Next steps:**
1. Integrate real TimesFM model for dynamic P10/P90 recalculation every 4h
2. Backtest on real historical Binance data (use ccxt or Binance API)
3. Add per-regime analysis (split metrics by range/trend/crash periods)
4. Optimize grid_levels, kelly_fraction, and circuit breaker thresholds via grid search
5. Test with different bar intervals (5-min, 15-min) for signal frequency tradeoff
6. Paper trade on Binance testnet before live deployment